<h1>Chapter 4 - Memory</h1>
<i>Exploring methodologies for remembering conversations</i>


<a href="..."><img src="https://img.shields.io/badge/Buy%20the%20Book!-grey?logo=amazon"></a>
<a href="..."><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="..."><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](...)

---

This notebook is for Chapter 4 of the [An Illustrated Guide to AI Agents](...) book by [Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/) and [Jay Alammar](https://www.linkedin.com/in/jalammar).

---

<a href="...">
<img src="https://learning.oreilly.com/covers/urn:orm:book:9798341662681/400w/" width="350"/></a>


### **[OPTIONAL]** - Installing Packages on Google Colab <img src="https://upload.wikimedia.org/wikipedia/commons/d/d0/Google_Colaboratory_SVG_Logo.svg" width=100>

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to **uncomment and run** one of the following codeblock to install the dependencies for this chapter. If you want to use a cloud provider, you only need to run the following code block:

In [ ]:
# %%capture
# !pip install illustrated-agents

---

💡 **NOTE**: If you want to use the GPU with `ollama`, then you will have to select a GPU first. In Google Colab, go to
**Runtime > Change runtime type > Hardware accelerator > GPU > GPU type > T4**. 

Then, **uncomment** and run this codeblock:

---

In [ ]:
# !apt-get install -y zstd > /dev/null 2>&1 && curl -fsSL https://ollama.com/install.sh | sh
# !nohup ollama serve > /dev/null 2>&1 & sleep 3 && ollama pull gemma3:12b && ollama pull embeddinggemma &

<hr style="height: 5px; border: none; border-radius: 5px; background: linear-gradient(to right, #000000, #7D7D7D);" />

## 1 - Choosing Your LLM

At the beginning of every chapter, we start by choosing the LLM that we want to use:

In [3]:
from openai import OpenAI
from illustrated_agents.chapters.ch2 import LLM

# Ollama through OpenAI API
client = OpenAI(base_url="http://localhost:11434/v1/", api_key="no_key")
llm = LLM(model="gemma3:12b", client=client)

# Llama.cpp server
# client = OpenAI(base_url="http://localhost:8080/v1/", api_key="no_key")
# llm = LLM(model="gemma-3-12B-it-Q4_K_M", client=client)

# LM Studio
# client = OpenAI(base_url="http://localhost:1234/v1/", api_key="no_key")
# llm = LLM(model="gemma-3-12B-it", client=client)

# Google's Gemini / Gemma
# client = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/", api_key="YOUR_GEMINI_API_KEY")
# llm = LLM(model="gemini-2.5-flash", client=client)

## 2 - Adding Retrieval Augmented Generation (RAG)


In the previous notebook, we added several forms of short-term memory (trimming and summarization). In this notebook we will cover Retrieval Augemented Generation (RAG) as a form of long-term memory.

We start by creating the EmbeddingModel class which uses the OpenAI-endpoint to query a given model. In our example, we are going to be using [EmbeddingGemma](https://ai.google.dev/gemma/docs/embeddinggemma) an embedding model with 308 million parameters. Quite a bit smaller than the 12 billion parameter LLM we have been using thus far!


In [ ]:
from openai import OpenAI


class EmbeddingModel:
    """Wrapper around an embedding model, mirroring the `LLM` class."""

    def __init__(self, model: str, client: OpenAI):
        self.model = model
        self.client = client

    def embed(self, text: str) -> list[float]:
        """Convert text into a numerical vector."""
        return self.client.embeddings.create(
            model=self.model, 
            input=text
        ).data[0].embedding


We then initialize this class and we can use the same client since we are going to be using Ollama as our backend. 

In [ ]:
# Initialize EmbeddingGemma
embedding_model = EmbeddingModel(model="embeddinggemma", client=client)

# Test the embedding model
output = embedding_model.embed("Dolphins are amazing!")
output

[-0.21747557818889618,
 -0.01567786931991577,
 0.03601466864347458,
 -0.0049857525154948235,
 -0.04253798723220825,
 -0.03880312666296959,
 -0.027158154174685478,
 0.03521708399057388,
 0.03940782696008682,
 -0.028296777978539467,
 -0.021544143557548523,
 -0.038480374962091446,
 -0.03742075711488724,
 -0.013370458036661148,
 0.06124311685562134,
 0.03188025578856468,
 0.015579869039356709,
 -0.0798254907131195,
 -0.04060091823339462,
 -0.025447947904467583,
 0.04106692597270012,
 0.029795804992318153,
 -0.008624075911939144,
 -0.0009055174305103719,
 0.0515257902443409,
 0.011594693176448345,
 -0.028225064277648926,
 0.04701607674360275,
 0.002130571287125349,
 0.025066230446100235,
 0.00023184649762697518,
 -0.0017705409554764628,
 0.013356722891330719,
 0.008962886407971382,
 -0.00291842850856483,
 0.029676716774702072,
 -0.05962586775422096,
 -0.03211178630590439,
 0.09139299392700195,
 -0.02481245808303356,
 -0.03459123522043228,
 0.05804670974612236,
 -0.01638619229197502,
 -0.019

The output is a list of 768 values, each between -1 and 1.

We can use these values to compare different documents and calculate their similarity. This is typically calculated as the cosine similarity, which represents the angle between embeddings. A smaller angle means a higher similarity. The cosine similarity is calculated through the dot product of the embeddings and then divided by the product of their lengths for normalization.

Let's try it out!

In [ ]:
from rich import print

# Create embeddings
embedding_a = embedding_model.embed("I love flamingos.")
embedding_b = embedding_model.embed("Dolphins use echolocation.")
embedding_c = embedding_model.embed("Flamingos are pink birds.")

# Calculate cosine similarity between A and B
dot_ab = sum(x * y for x, y in zip(embedding_a, embedding_b))
norm_a = sum(x * x for x in embedding_a) ** 0.5
norm_b = sum(x * x for x in embedding_b) ** 0.5
similarity_ab = dot_ab / (norm_a * norm_b)

# Calculate cosine similarity between A and C
dot_ac = sum(x * y for x, y in zip(embedding_a, embedding_c))
norm_c = sum(x * x for x in embedding_c) ** 0.5
similarity_ac = dot_ac / (norm_a * norm_c)

print(f"Similarity between A and B: {similarity_ab}")
print(f"Similarity between A and C: {similarity_ac}")

Similarity between A and B: 0.3546779285862413

Similarity between A and C: 0.6386974942657335






As such, the `RAGMemory` that we are going to implement has the following steps:

1) Embed all external documents the Agent has no direct access to (`__init__`)
2) Embed the user's query (`embedding_model.embed(query)`)
3) Compared the embeddings and create a similarity matrix (`.`)
4) Return the documents with the highest similarity
5) Add those documents to the prompt

In [ ]:
from illustrated_agents.chapters.ch4 import Memory

class RAGMemory(Memory):
    """Long-term memory with RAG."""

    def __init__(self, embedding_model: EmbeddingModel, documents: list[str]):
        super().__init__()
        self.embedding_model = embedding_model
        self.documents = documents
        self.embeddings = [embedding_model.embed(doc) for doc in documents]

    def add(self, role: str, content: str):
        # Augment user queries with retrieved context before storing
        if role == "user":
            context = "\n".join(self.search(content))
            content = f"""Context:
{context}

Question: {content}"""
        super().add(role, content)

    def search(self, query: str) -> list[str]:
        """Return the top-k documents most similar to the query."""
        query_embed = self.embedding_model.embed(query)
        scores = [self._cosine(query_embed, embed) for embed in self.embeddings]
        ranked = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
        return [self.documents[index] for index in ranked[:3]]

    def _cosine(self, a: list[float], b: list[float]) -> float:
        """Calculate cosine similarity between two embeddings.."""
        return sum(x * y for x, y in zip(a, b))

Let’s put this to practice, starting with a set of documents that your TinyAgent has no direct access to. This is going to be a simple example, but imagine you have thousands of documents.

In [ ]:
from illustrated_agents.chapters.ch4 import TinyAgent

# RAGMemory with external documents
documents = [
    "Sarah works as a marine biologist studying coral reefs.",
    "Sarah lives in Lisbon, Portugal.",
    "Sarah's favorite hobby is rock climbing.",
    "Sarah favorite animals are flamingos.",
    "Sarah speaks fluent Spanish and Portuguese.",
    "Ilse is a software engineer at a renewable energy startup.",
    "Ilse lives in Amsterdam, the Netherlands.",
    "Ilse plays the cello in a local string quartet.",
    "Ilse's favorite author is Brandon Sanderson.",
    "Ilse's favorite animals are dolphins.",
]
memory = RAGMemory(documents=documents, embedding_model=embedding_model)

# Create the Agent and run a query
agent = TinyAgent(llm=llm, memory=memory)
response = agent.run("What is Sarah's favorite animal?")
print(response)

Sarah's favorite animal is flamingos.


In [ ]:
print(agent.memory.get_messages())

[
    {
        'role': 'user',
        'content': "Context:\nSarah favorite animals are flamingos.\nSarah's favorite hobby is rock 
climbing.\nIlse's favorite animals are dolphins.\n\nQuestion: What is Sarah's favorite animal?"
    },
    {'role': 'assistant', 'content': "Sarah's favorite animal is flamingos."}
]

Note how it retrieved the top 3 documents as the context? This is both the advantage and disadvantage of RAG. Although it minimizes the context that you have to pass to the model, there is no guarantee that the context will always be good enough. You could, for example, only accept documents that have a minimum degree of similarity rather than simply getting the top 3 irrespective of their absolute scores.

<hr style="height: 5px; border: none; border-radius: 5px; background: linear-gradient(to right, #000000, #7D7D7D);" />

# What We Built

In this chapter, we covered how `Memory` could be added to your `TinyAgent`. There are now three main concepts in total (LLM, Memory, and TinyAgent):

In [1]:
from illustrated_agents.chapters.ch4_long_term_memory import what_we_built; what_we_built

╭───────────────────────────────────────────────── What We Built ─────────────────────────────────────────────────╮
│ TinyAgent                                                                                                       │
│ ├── agent.py                                                                                                    │
│ ├── llm.py        ← Updated (Add `EmbeddingModel` to support long-term memory.)                                 │
│ ├── memory.py     ← Updated (Add `LongTermMemory` to the TinyAgent in the form of RAG.)                         │
│ └── trajectory.py                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯